# Chinook NL2SQL Agent (Notebook)

基于 `chinook.db` 的规则型问答 Agent（NL2SQL）。

已覆盖示例问题：
- 数据库中总共有多少张表；
- 员工表中有多少条记录；
- 在数据库中所有客户个数和员工个数分别是多少。


In [3]:
from __future__ import annotations

import re
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple


@dataclass
class AgentResult:
    question: str
    sql: str
    rows: List[Tuple]
    answer: str


class NL2SQLAgent:
    TABLE_CN_TO_EN: Dict[str, str] = {
        '专辑': 'albums',
        '艺术家': 'artists',
        '客户': 'customers',
        '员工': 'employees',
        '流派': 'genres',
        '发票': 'invoices',
        '发票明细': 'invoice_items',
        '媒体类型': 'media_types',
        '播放列表': 'playlists',
        '播放列表曲目': 'playlist_track',
        '曲目': 'tracks',
    }

    def __init__(self, db_path: str | Path) -> None:
        self.db_path = Path(db_path)
        if not self.db_path.exists():
            raise FileNotFoundError(f'Database file not found: {self.db_path}')
        self.conn = sqlite3.connect(str(self.db_path))

    def close(self) -> None:
        self.conn.close()

    def ask(self, question: str) -> AgentResult:
        sql = self._question_to_sql(question)
        rows = self._run_sql(sql)
        answer = self._rows_to_answer(sql, rows)
        return AgentResult(question=question, sql=sql, rows=rows, answer=answer)

    def _run_sql(self, sql: str) -> List[Tuple]:
        cur = self.conn.cursor()
        cur.execute(sql)
        return cur.fetchall()

    def _question_to_sql(self, question: str) -> str:
        q = self._normalize(question)

        if ('多少张表' in q or '表数量' in q or '表个数' in q) and ('数据库' in q or '库中' in q):
            return (
                "SELECT COUNT(*) AS table_count "
                "FROM sqlite_master "
                "WHERE type='table' AND name NOT LIKE 'sqlite_%';"
            )

        if ('员工表' in q or 'employees' in q) and self._has_count_intent(q):
            return 'SELECT COUNT(*) AS employee_count FROM employees;'

        if ('客户' in q or 'customers' in q) and ('员工' in q or 'employees' in q) and self._has_count_intent(q):
            return (
                "SELECT "
                "(SELECT COUNT(*) FROM customers) AS customer_count, "
                "(SELECT COUNT(*) FROM employees) AS employee_count;"
            )

        matched_table = self._extract_table_name(q)
        if matched_table and self._has_count_intent(q):
            return f'SELECT COUNT(*) AS record_count FROM {matched_table};'

        raise ValueError(f'暂不支持该问题：{question}')

    @staticmethod
    def _normalize(question: str) -> str:
        q = question.strip().lower()
        q = q.replace('？', '?').replace('。', '.')
        q = re.sub(r'\s+', '', q)
        return q

    @staticmethod
    def _has_count_intent(q: str) -> bool:
        return any(x in q for x in ['多少', '几', '总数', '个数', '数量', '记录数', '条记录', '多少条', '多少行'])

    def _extract_table_name(self, q: str) -> str | None:
        for cn_name, en_name in sorted(self.TABLE_CN_TO_EN.items(), key=lambda x: len(x[0]), reverse=True):
            if f'{cn_name}表' in q or f'{cn_name}' in q:
                return en_name

        for en_name in self.TABLE_CN_TO_EN.values():
            if en_name in q:
                return en_name

        return None

    def _rows_to_answer(self, sql: str, rows: Sequence[Tuple]) -> str:
        if 'sqlite_master' in sql:
            return f'数据库中总共有 {rows[0][0]} 张表。'

        if 'customer_count' in sql and 'employee_count' in sql:
            customer_count, employee_count = rows[0]
            return f'数据库中客户个数为 {customer_count}，员工个数为 {employee_count}。'

        if 'from employees' in sql.lower() and 'employee_count' in sql.lower():
            return f'员工表中共有 {rows[0][0]} 条记录。'

        if rows and len(rows[0]) == 1:
            return f'查询结果为 {rows[0][0]}。'

        return f'查询结果：{rows}'


In [4]:
# 示例运行
db_path = Path('chinook.db')
agent = NL2SQLAgent(db_path)

questions = [
    '数据库中总共有多少张表',
    '员工表中有多少条记录',
    '在数据库中所有客户个数和员工个数分别是多少',
]

for idx, question in enumerate(questions, 1):
    result = agent.ask(question)
    print(f'提问{idx}: {result.question}')
    print(f'SQL: {result.sql}')
    print(f'回答: {result.answer}')
    print('-' * 70)

agent.close()


提问1: 数据库中总共有多少张表
SQL: SELECT COUNT(*) AS table_count FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';
回答: 数据库中总共有 11 张表。
----------------------------------------------------------------------
提问2: 员工表中有多少条记录
SQL: SELECT COUNT(*) AS employee_count FROM employees;
回答: 员工表中共有 8 条记录。
----------------------------------------------------------------------
提问3: 在数据库中所有客户个数和员工个数分别是多少
SQL: SELECT (SELECT COUNT(*) FROM customers) AS customer_count, (SELECT COUNT(*) FROM employees) AS employee_count;
回答: 数据库中客户个数为 59，员工个数为 8。
----------------------------------------------------------------------
